# USD/VND Exchange Rate — Data Pipeline

### `USD_VND_Cleaned_Features.csv`
`Date`, `USD_VND`, `Price`, `Change %`, `Lag_1`, `Lag_5`, `ROC_5`, `Volatility_1_Week`, `DXY`, `US_Bond_10Y`, `US_Fed_Rate`, `VN_Interbank_Rate`, `VN_CPI`, `Trade_Balance`, `FDI_Disbursed`, `Month`, `DayOfWeek`

### `USD_VND_Model_Ready.csv`
`Lag_1`, `Lag_5`, `ROC_5`, `Volatility_1_Week`, `DXY`, `US_Bond_10Y`, `US_Fed_Rate`, `VN_Interbank_Rate`, `VN_CPI`, `Trade_Balance`, `FDI_Disbursed`, `Month`, `DayOfWeek`

FRED macro columns are populated when online; kept as `NaN` when offline.


In [11]:
import io, time, warnings, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
print("Libraries loaded ✓")


Libraries loaded ✓


## Section 1 — Raw Data
Loads `usd_vnd_history.csv` if present (from Playwright scraper). Falls back to synthetic data otherwise.

In [12]:
RAW_CSV = Path("usd_vnd_history.csv")

def make_synthetic_data(start="2022-01-01", end="2026-03-06") -> pd.DataFrame:
    rng     = np.random.default_rng(42)
    dates   = pd.bdate_range(start=start, end=end)
    n       = len(dates)
    returns = rng.normal(loc=0.00005, scale=0.003, size=n)
    close   = 23_000 * np.exp(np.cumsum(returns))
    change  = (close / np.roll(close, 1) - 1) * 100
    change[0] = 0.0
    return pd.DataFrame({
        "Date":     dates.strftime("%d/%m/%Y"),
        "Price":    np.round(close, 1),
        "Change %": np.round(change, 2),
    })

if RAW_CSV.exists():
    raw_df = pd.read_csv(RAW_CSV)
    print(f"Loaded {RAW_CSV}  ({len(raw_df)} rows)")
else:
    print("usd_vnd_history.csv not found — generating synthetic data…")
    raw_df = make_synthetic_data()
    raw_df.to_csv(RAW_CSV, index=False)
    print(f"Synthetic data saved -> {RAW_CSV}  ({len(raw_df)} rows)")

raw_df.tail(3)


Loaded usd_vnd_history.csv  (1092 rows)


,Date,Price,Open,High,Low,Change %
1089,05/01/2022,"22,755.0","22,750.0","22,770.0","22,735.0",+0.01%
1090,04/01/2022,"22,752.0","22,770.0","22,782.5","22,750.0",-0.32%
1091,03/01/2022,"22,825.0","22,825.0","22,825.0","22,825.0",0.00%


## Section 2 — Clean & Format

In [20]:
def clean_raw(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "Price" in df.columns:
        df["Price"] = (
            df["Price"].astype(str)
                       .str.replace('"', "", regex=False)
                       .str.replace(",", "", regex=False)
                       .pipe(pd.to_numeric, errors="coerce")
        )

    if "Change %" in df.columns:
        df["Change %"] = (
            df["Change %"].astype(str)
                          .str.replace('"', "", regex=False)
                          .str.replace("%", "", regex=False)
                          .str.replace("+", "", regex=False)
                          .pipe(pd.to_numeric, errors="coerce")
        )

    for fmt in ("%d/%m/%Y", "%Y-%m-%d", None):
        try:
            df["Date"] = pd.to_datetime(df["Date"], format=fmt, errors="raise")
            break
        except Exception:
            continue

    df = df.dropna(subset=["Date"])
    df = df[df["Date"].dt.dayofweek < 5]   # business days only
    df = df.sort_values("Date").reset_index(drop=True)
    df["USD_VND"] = df["Price"]

    print(f"Rows: {len(df)}  |  {df['Date'].min().date()} -> {df['Date'].max().date()}")
    return df

df = clean_raw(raw_df)
df.tail(3)


Rows: 1090  |  2022-01-03 -> 2026-03-06


,Date,Price,Open,High,Low,Change %,USD_VND
1087,2026-03-04,26220.0,"26,244.5","26,255.0","26,181.0",0.08,26220.0
1088,2026-03-05,26215.0,"26,207.0","26,256.5","26,170.0",-0.02,26215.0
1089,2026-03-06,26240.0,"26,228.0","26,250.0","26,215.0",0.10,26240.0


## Section 3 — Feature Engineering

In [21]:
price = df["Price"]

df["Lag_1"]             = price.shift(1)
df["Lag_5"]             = price.shift(5)
df["ROC_5"]             = price.pct_change(5) * 100
df["Volatility_1_Week"] = price.rolling(5).std()
df["Month"]             = df["Date"].dt.month
df["DayOfWeek"]         = df["Date"].dt.dayofweek   # 0=Mon … 4=Fri

print("Features added ✓")
df[["Date","Price","Change %","Lag_1","Lag_5","ROC_5","Volatility_1_Week","Month","DayOfWeek"]].tail(5)


Features added ✓


,Date,Price,Change %,Lag_1,Lag_5,ROC_5,Volatility_1_Week,Month,DayOfWeek
1085,2026-03-02,26165.0,0.46,26045.0,26120.0,0.172282,62.368261,3,0
1086,2026-03-03,26200.0,0.13,26165.0,26195.0,0.019088,63.963271,3,1
1087,2026-03-04,26220.0,0.08,26200.0,26102.0,0.452073,77.249595,3,2
1088,2026-03-05,26215.0,-0.02,26220.0,26075.0,0.536913,72.577545,3,3
1089,2026-03-06,26240.0,0.10,26215.0,26045.0,0.748704,27.973201,3,4


## Section 4 — Macro Features (FRED)
Fetches 7 macro series. **Skipped gracefully when offline** — columns remain as `NaN`.

In [22]:
FRED_BASE   = "https://fred.stlouisfed.org/graph/fredgraph.csv?id="
FRED_SERIES = {
    "DXY"              : ["DTWEXBGS", "DTWEXM"],
    "US_Bond_10Y"      : ["DGS10", "GS10"],
    "US_Fed_Rate"      : ["FEDFUNDS", "DFEDTARL"],
    "VN_CPI"           : ["FPCPITOTLZGVNM"],
}

def fetch_fred(series_id: str, col: str, retries: int = 2) -> pd.DataFrame:
    url = FRED_BASE + series_id
    for attempt in range(1, retries + 1):
        try:
            with urllib.request.urlopen(url, timeout=15) as resp:
                raw = resp.read()
            tmp = pd.read_csv(io.StringIO(raw.decode()))
            tmp.columns = ["Date", col]
            tmp["Date"] = pd.to_datetime(tmp["Date"], errors="coerce")
            tmp[col]    = pd.to_numeric(tmp[col], errors="coerce")
            return tmp.dropna(subset=["Date"]).sort_values("Date")
        except Exception as exc:
            if attempt == retries:
                print(f"  [SKIP] {series_id}: {exc}")
            else:
                time.sleep(2 ** attempt)
    return pd.DataFrame(columns=["Date", col])

def fetch_best_fred(candidates: list, col: str) -> pd.DataFrame:
    for sid in candidates:
        result = fetch_fred(sid, col)
        if not result.empty and result[col].notna().any():
            print(f"  [OK] {col} <- FRED:{sid}  ({len(result)} rows)")
            return result
    return pd.DataFrame(columns=["Date", col])

def merge_macro(main: pd.DataFrame, macro: pd.DataFrame, col: str) -> pd.DataFrame:
    if macro.empty or col not in macro.columns:
        main[col] = np.nan
        return main
    merged = main.merge(macro[["Date", col]], on="Date", how="left")
    merged[col] = pd.to_numeric(merged[col], errors="coerce").ffill()
    return merged

print("Fetching FRED macro series…")
for col_name, series_list in FRED_SERIES.items():
    mdf = fetch_best_fred(series_list, col_name)
    df  = merge_macro(df, mdf, col_name)

populated = [c for c in FRED_SERIES if df[c].notna().any()]
skipped   = [c for c in FRED_SERIES if not df[c].notna().any()]
print(f"Populated : {populated or 'none (offline)'}")
if skipped:
    print(f"Skipped   : {skipped} -> NaN")


Fetching FRED macro series…
  [OK] DXY <- FRED:DTWEXBGS  (5265 rows)
  [OK] US_Bond_10Y <- FRED:DGS10  (16747 rows)
  [OK] US_Fed_Rate <- FRED:FEDFUNDS  (860 rows)
  [OK] VN_CPI <- FRED:FPCPITOTLZGVNM  (29 rows)
Populated : ['DXY', 'US_Bond_10Y', 'US_Fed_Rate', 'VN_CPI']


## Section 5 — Select Final Columns & Save

In [23]:
CLEANED_COLS = [
    "Date", "USD_VND", "Price", "Change %",
    "Lag_1", "Lag_5", "ROC_5", "Volatility_1_Week",
    "DXY", "US_Bond_10Y", "US_Fed_Rate",
    "VN_CPI", "Month", "DayOfWeek",
]

MODEL_COLS = [
    "Lag_1", "Lag_5", "ROC_5", "Volatility_1_Week",
    "DXY", "US_Bond_10Y", "US_Fed_Rate",
    "VN_CPI", "Month", "DayOfWeek",
]

df_clean = df[CLEANED_COLS].dropna(subset=["Lag_5", "Volatility_1_Week"]).reset_index(drop=True)
df_model = df_clean[MODEL_COLS].copy()

df_clean.to_csv("USD_VND_Cleaned_Features.csv", index=False)
df_model.to_csv("USD_VND_Model_Ready.csv",      index=False)

print(f"USD_VND_Cleaned_Features.csv  ({len(df_clean)} rows, {df_clean.shape[1]} cols)")
print(f"USD_VND_Model_Ready.csv       ({len(df_model)} rows, {df_model.shape[1]} cols)")

summary = pd.DataFrame({
    "column":  list(df_clean.columns),
    "nulls_%": (df_clean.isna().mean() * 100).round(1).values,
})
print(summary.to_string(index=False))
df_clean.tail(5)


USD_VND_Cleaned_Features.csv  (1085 rows, 14 cols)
USD_VND_Model_Ready.csv       (1085 rows, 10 cols)
           column  nulls_%
             Date      0.0
          USD_VND      0.0
            Price      0.0
         Change %      0.0
            Lag_1      0.0
            Lag_5      0.0
            ROC_5      0.0
Volatility_1_Week      0.0
              DXY      0.0
      US_Bond_10Y      0.0
      US_Fed_Rate      1.5
           VN_CPI     47.5
            Month      0.0
        DayOfWeek      0.0


,Date,USD_VND,Price,Change %,Lag_1,Lag_5,ROC_5,Volatility_1_Week,DXY,US_Bond_10Y,US_Fed_Rate,VN_CPI,Month,DayOfWeek
1080,2026-03-02,26165.0,26165.0,0.46,26045.0,26120.0,0.172282,62.368261,118.6670,4.05,3.64,3.621093,3,0
1081,2026-03-03,26200.0,26200.0,0.13,26165.0,26195.0,0.019088,63.963271,119.4341,4.06,3.64,3.621093,3,1
1082,2026-03-04,26220.0,26220.0,0.08,26200.0,26102.0,0.452073,77.249595,119.0705,4.09,3.64,3.621093,3,2
1083,2026-03-05,26215.0,26215.0,-0.02,26220.0,26075.0,0.536913,72.577545,119.5683,4.13,3.64,3.621093,3,3
1084,2026-03-06,26240.0,26240.0,0.10,26215.0,26045.0,0.748704,27.973201,119.4910,4.15,3.64,3.621093,3,4
